# CSC 2301 · Week 12 — Algorithms: searching, sorting, interpolating

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/csc2301/notebooks/CSC2301_W12_algorithms_search_interp.ipynb)

**IUGB · BSc Mining / Petroleum Engineering** · Instructor: Laurent Kouadio

An **algorithm** is a precise recipe that a computer can follow. Python already has `sorted`, `bisect`
and `np.interp`, but an engineer must know **how** they work and **how fast** they are, to choose well
and to trust the result. This week you write the classic ones yourself and compare them with the library.
The data are the week 8 log of well KW-2 and a temperature survey of geothermal well GT-1 (`temperature_survey_gt1.csv`,
synthetic), measured at irregular depths.

**What you will practise:** linear search · binary search · counting steps (complexity) ·
selection sort · linear interpolation · comparing your code with the library.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/csc2301/'
FILES = [('data/temperature_survey_gt1.csv', 'temperature_survey_gt1.csv'), ('data/well_log_kw2.csv', 'well_log_kw2.csv'), ('tests/w12/q1.py', 'tests/q1.py'), ('tests/w12/q2.py', 'tests/q2.py'), ('tests/w12/q3.py', 'tests/q3.py'), ('tests/w12/q4.py', 'tests/q4.py'), ('tests/w12/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

## 1. How many steps?

To find a value in a list, a **linear search** looks at each element in turn: up to *n* steps.
If the list is **sorted** (depths always are), a **binary search** looks at the middle, throws away the
half that cannot contain the answer, and repeats: about log₂(*n*) steps. For a million samples:
a million steps against twenty.

In [ ]:
import math

for n in [10, 1_000, 1_000_000, 1_000_000_000]:
    print(f"n = {n:>13,}   linear: up to {n:>13,} steps   binary: about {math.ceil(math.log2(n)):>2} steps")

## 2. Binary search by hand

Keep a window `[lo, hi)` that surely contains the answer, and halve it until it is empty.

In [ ]:
depths = [100, 250, 400, 550, 700, 850, 1000]
target = 620
lo, hi = 0, len(depths)
while lo < hi:
    mid = (lo + hi) // 2
    print(f"lo={lo} hi={hi} mid={mid} depths[mid]={depths[mid]}")
    if depths[mid] < target:
        lo = mid + 1          # the answer is to the right of mid
    else:
        hi = mid              # mid may be the answer: keep it
print("first depth >=", target, "is at index", lo, "->", depths[lo])

## 3. The data

In [ ]:
import csv
import numpy as np

log = np.loadtxt("well_log_kw2.csv", delimiter=",", skiprows=1)
log_depth = list(log[:, 0])
gr = list(log[:, 1])

with open("temperature_survey_gt1.csv", encoding="utf-8") as f:
    survey = list(csv.DictReader(f))
depths = [float(r["depth_m"]) for r in survey]
temps = [float(r["temp_c"]) for r in survey]
print(len(gr), "log samples;", len(depths), "temperature stations")
print(list(zip(depths, temps))[:5])

---
## Exercises

Write the algorithms with loops; the checks compare your answers with the library versions.

### Q1. Linear search

Write `find_first(values, threshold)` that returns the **position** of the first value greater than
`threshold`, or `-1` if there is none. Then store in `first_shale` the index of the first GR sample above
100 API in well KW-2. What depth is it?

In [ ]:
def find_first(values, threshold):
    ...

first_shale = ...
first_shale, log_depth[first_shale]

In [ ]:
grader.check("q1")

### Q2. Binary search

Write `depth_index(depths, d)` for a **sorted** list: it returns the index of the first depth `>= d`
(0 if `d` is above the first depth, `len(depths)` if `d` is deeper than the last). Use the `lo` / `hi` method
of section 2. Check yourself against `bisect.bisect_left(depths, d)`.

In [ ]:
def depth_index(depths, d):
    ...

import bisect
depth_index(log_depth, 2250.2), bisect.bisect_left(log_depth, 2250.2)

In [ ]:
grader.check("q2")

### Q3. Selection sort

Write `selection_sort(values)` that returns a **new** list sorted from smallest to largest, without changing
`values` and without `sorted`, `.sort()` or `min`. Method: for each position `i`, find the smallest value in
`i…end`, and swap it into position `i`.

In [ ]:
def selection_sort(values):
    result = values[:]       # a copy
    ...
    return result

selection_sort([2.31, 2.05, 2.62, 2.05, 1.98])

In [ ]:
grader.check("q3")

### Q4. Linear interpolation

Between two measured points $(x_0, y_0)$ and $(x_1, y_1)$, the straight-line estimate at $x$ is

$$y = y_0 + (x - x_0)\,\frac{y_1 - y_0}{x_1 - x_0}$$

Write `interp(x, x0, y0, x1, y1)`.

In [ ]:
def interp(x, x0, y0, x1, y1):
    ...

interp(5, 0, 10, 10, 30)    # 20.0

In [ ]:
grader.check("q4")

### Q5. Temperature at any depth

Write `temperature_at(depth, depths, temps)` that estimates the temperature at `depth` from the survey:
find the surrounding stations with `depth_index`, then use `interp`. A depth equal to a station gives that
station's temperature; a depth outside the survey raises `ValueError`. Store in `t_2250` the temperature at
2250 m, and compare with `np.interp(2250, depths, temps)`.

In [ ]:
def temperature_at(depth, depths, temps):
    ...

t_2250 = ...
t_2250, np.interp(2250, depths, temps)

In [ ]:
grader.check("q5")

### Going further (not checked)

* Count the comparisons made by `selection_sort` for 10, 100 and 1000 values. How does the count grow?
  Time it against `sorted` with `%timeit`.
* Fit a straight line to the survey with `np.polyfit(depths, temps, 1)`: what is the geothermal gradient
  in °C/km? Is one line enough, or are there two?

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()